# Task 4：银行股滚动回测

对应 [4-银行股滚动回测.md](./4-银行股滚动回测.md)。

通过滚动窗口方法评估定投策略对起点的敏感性，对每只银行计算 4 项胜率指标：XIRR 为正、跑赢逆回购、跑赢沪深300全收益、跑赢中证500全收益。基准同样使用等额定投口径。

## 1. 环境与参数

滚动窗口：期限 3/5/10 年，步长 1 个月。每月投入 5,000 元，1 日买入，100 股整数倍，分红再投资，不计手续费和分红税。

In [ ]:
from __future__ import annotations

import importlib
import math
import os
import re
import socket
import sys
import time
from contextlib import contextmanager
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Callable, Iterable
from unittest.mock import patch

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


WORKING_DIR = Path.cwd().resolve()
LABS_DIR = None
for _candidate in [WORKING_DIR, *WORKING_DIR.parents]:
    if _candidate.name == "labs" and (_candidate / "pyproject.toml").is_file():
        LABS_DIR = _candidate
        break
    if (_candidate / "labs" / "pyproject.toml").is_file():
        LABS_DIR = _candidate / "labs"
        break

LAB_DIR = LABS_DIR / "01_银行股定投回测"


plt.rcParams["font.sans-serif"] = [
    "Microsoft YaHei", "SimHei", "Arial Unicode MS", "DejaVu Sans"
]
plt.rcParams["axes.unicode_minus"] = False


AS_OF_DATE = "2026-07-31"
HORIZONS = (3, 5, 10)
MONTHLY_AMOUNT = 5000
BUY_DAY = 1
LOT_SIZE = 100
DIVIDEND_REINVEST = True
MIN_COMPARABLE_WINDOWS = 12

FIXED_SAMPLE = [
    ("601398", "工商银行"),
    ("601939", "建设银行"),
    ("600036", "招商银行"),
    ("601288", "农业银行"),
    ("601166", "兴业银行"),
]

DATA_DIR = LAB_DIR / "data" / "lab2"
PRICE_DIR = DATA_DIR / "prices"
DIVIDEND_DIR = DATA_DIR / "dividends"
INDEX_DIR = DATA_DIR / "index_prices"
CHART_DIR = LAB_DIR / "data" / "charts" / "rolling"
RESULT_DIR = LAB_DIR / "data" / "results"
for path in (CHART_DIR, RESULT_DIR, INDEX_DIR):
    path.mkdir(parents=True, exist_ok=True)

print({"AS_OF_DATE": AS_OF_DATE, "HORIZONS": HORIZONS, "MIN_COMPARABLE_WINDOWS": MIN_COMPARABLE_WINDOWS})

## 2. 函数定义

以下函数从原 `bank_core.py` 内联到本 Notebook。先定义回测计算函数（与 Task 2/3 共用），再定义滚动窗口函数和指数行情工具。

### 回测计算函数

本组函数实现月度定投回测的核心逻辑：构建标的总收益净值、计算 XIRR、执行按月买入和分红再投资。

- `normalize_symbol`：从任意输入提取 6 位证券代码，前补零。
- `build_total_return_history`：用不复权收盘价 + 每股现金分红构建标的总收益净值（用于回撤计算）。
- `xirr`：计算 XIRR（内部收益率），牛顿迭代法，容忍不规则现金流日期。
- `contribution_dates`：计算定投买入日（每月 1 日，非交易日顺延至下一个交易日）。
- `_shares_on_or_before`：查找登记日对应的持股快照（用于分红再投资时的持股数）。
- `_max_loss_duration_days`：账户资产低于累计投入的最长连续天数。
- `_strategy_max_drawdown`：现金流调整后策略净值（含分红再投资）的最大回撤。
- `BacktestOutput`：数据类，封装回测结果（summary/transactions/account_history/total_return_history）。
- `simulate_bank_dca`：单只银行月度定投回测主函数：按月买入、分红再投资、计算 9 组指标。

In [ ]:
def normalize_symbol(value: Any) -> str:
    digits = "".join(c for c in str(value) if c.isdigit())
    return digits[-6:].zfill(6)


def build_total_return_history(
    prices: pd.DataFrame,
    dividends: pd.DataFrame,
) -> pd.DataFrame:
    """用不复权收盘价和每股现金分红构建标的总收益净值。"""
    frame = prices[["date", "close"]].copy()
    frame["date"] = pd.to_datetime(frame["date"])
    frame = frame.dropna().drop_duplicates("date", keep="last").sort_values("date")
    events = dividends[["ex_date", "cash_dividend_per_share"]].copy()
    events["ex_date"] = pd.to_datetime(events["ex_date"])
    events = events.groupby("ex_date", as_index=False)["cash_dividend_per_share"].sum()
    frame = frame.merge(events, left_on="date", right_on="ex_date", how="left")
    frame["cash_dividend_per_share"] = frame["cash_dividend_per_share"].fillna(0.0)
    frame["daily_total_return"] = (
        (frame["close"] + frame["cash_dividend_per_share"])
        / frame["close"].shift(1)
        - 1.0
    )
    frame.loc[frame.index[0], "daily_total_return"] = 0.0
    frame["total_return_nav"] = (1.0 + frame["daily_total_return"]).cumprod()
    frame["drawdown"] = (
        frame["total_return_nav"] / frame["total_return_nav"].cummax() - 1.0
    )
    return frame[
        [
            "date", "close", "cash_dividend_per_share", "daily_total_return",
            "total_return_nav", "drawdown",
        ]
    ]


def xirr(cashflows: Iterable[float], dates: Iterable[Any]) -> float:
    values = np.asarray(list(cashflows), dtype=float)
    timestamps = [pd.Timestamp(date) for date in dates]
    if len(values) != len(timestamps) or len(values) < 2:
        return np.nan
    if np.all(values >= 0) or np.all(values <= 0):
        return np.nan
    years = np.array(
        [(date - timestamps[0]).total_seconds() / (365.25 * 86400)
         for date in timestamps],
        dtype=float,
    )

    def npv(rate: float) -> float:
        return float(np.sum(values / np.power(1.0 + rate, years)))

    lower = -0.9999
    upper = 1.0
    lower_value = npv(lower)
    upper_value = npv(upper)
    while lower_value * upper_value > 0 and upper < 1_000_000:
        upper = upper * 2.0 + 1.0
        upper_value = npv(upper)
    if lower_value * upper_value > 0:
        return np.nan
    for _ in range(250):
        midpoint = (lower + upper) / 2.0
        midpoint_value = npv(midpoint)
        if abs(midpoint_value) < 1e-8:
            return float(midpoint)
        if lower_value * midpoint_value <= 0:
            upper = midpoint
        else:
            lower = midpoint
            lower_value = midpoint_value
    return float((lower + upper) / 2.0)


def contribution_dates(
    trading_dates: Iterable[Any],
    *,
    start_date: Any,
    end_date: Any,
    buy_day: int = 1,
) -> list[pd.Timestamp]:
    dates = pd.DatetimeIndex(pd.to_datetime(list(trading_dates))).sort_values().unique()
    start = pd.Timestamp(start_date)
    end = pd.Timestamp(end_date)
    dates = dates[(dates >= start) & (dates <= end)]
    if len(dates) == 0:
        return []

    selected: list[pd.Timestamp] = [pd.Timestamp(dates[0])]
    first_period = pd.Timestamp(dates[0]).to_period("M")
    last_period = pd.Timestamp(dates[-1]).to_period("M")
    for period in pd.period_range(first_period + 1, last_period, freq="M"):
        target = period.start_time + pd.Timedelta(days=buy_day - 1)
        candidates = dates[
            (dates.to_period("M") == period) & (dates >= target)
        ]
        if len(candidates):
            selected.append(pd.Timestamp(candidates[0]))
    return selected


def _shares_on_or_before(
    snapshots: dict[pd.Timestamp, int],
    date: pd.Timestamp,
) -> int:
    eligible = [key for key in snapshots if key <= date]
    return snapshots[max(eligible)] if eligible else 0


def _max_loss_duration_days(account_history: pd.DataFrame) -> int:
    """账户资产低于累计投入的最长连续天数。"""
    if account_history.empty:
        return 0
    asset = account_history["account_asset"].astype(float).reset_index(drop=True)
    contribution = account_history["cumulative_contribution"].astype(float).reset_index(drop=True)
    underwater = (asset < contribution).fillna(False)
    if not underwater.any():
        return 0
    max_run = 0
    current_run = 0
    for flag in underwater:
        if flag:
            current_run += 1
            if current_run > max_run:
                max_run = current_run
        else:
            current_run = 0
    return int(max_run)


def _strategy_max_drawdown(account_history: pd.DataFrame) -> float:
    """现金流调整后策略净值回撤：剔除新增本金的影响。"""
    if account_history.empty:
        return np.nan
    data = account_history.copy()
    data["net_value"] = (
        data["account_asset"] - data["cumulative_contribution"].shift(1).fillna(0)
    )
    # 起点净值为 0，无法直接计算回撤，使用累计净流入作为基线
    data["strategy_nav"] = data["net_value"].cummax().where(
        data["net_value"] > 0, other=data["net_value"]
    )
    data["strategy_drawdown"] = (
        data["strategy_nav"] / data["strategy_nav"].cummax() - 1.0
    )
    return float(data["strategy_drawdown"].min())


@dataclass
class BacktestOutput:
    summary: dict[str, Any]
    transactions: pd.DataFrame
    account_history: pd.DataFrame
    total_return_history: pd.DataFrame


def simulate_bank_dca(
    *,
    symbol: str,
    name: str,
    prices: pd.DataFrame,
    dividends: pd.DataFrame,
    listing_date: Any,
    as_of_date: str,
    horizon_years: int,
    monthly_amount: float = 5000.0,
    buy_day: int = 1,
    lot_size: int = 100,
    dividend_reinvest: bool = True,
) -> BacktestOutput:
    """单只银行月度定投回测。

    纯计算函数：输入不复权行情 + 已实施分红 DataFrame，输出交易流水、
    账户历史、标的总收益净值与汇总指标。不调用任何外部接口。

    回测口径（与本 Lab .md 一致）：
    - 每月投入固定金额，不足 1 手的零钱留现金；
    - 除权日按收盘价立即用分红再投资原标的；
    - 不计算手续费与分红税；
    - 上市晚于窗口起点或数据不完整时，比较指标返回 np.nan。
    """
    symbol = normalize_symbol(symbol)
    as_of = pd.Timestamp(as_of_date)
    requested_start = as_of - pd.DateOffset(years=horizon_years)
    listing = pd.Timestamp(listing_date)

    price = prices.copy()
    price["date"] = pd.to_datetime(price["date"])
    price = price[
        price["date"].le(as_of) & price["date"].ge(min(requested_start, listing))
    ].drop_duplicates("date", keep="last").sort_values("date")
    if price.empty:
        raise ValueError(f"{symbol} 在 {horizon_years} 年窗口内无行情")

    actual_start_target = max(requested_start, listing)
    available = price[price["date"].ge(actual_start_target)]
    if available.empty:
        raise ValueError(f"{symbol} 上市后至截止日无行情")
    start = pd.Timestamp(available["date"].iloc[0])
    end = pd.Timestamp(price["date"].max())
    price = price[price["date"].between(start, end)].reset_index(drop=True)
    full_horizon = bool(
        listing <= requested_start
        and start <= requested_start + pd.Timedelta(days=15)
    )

    schedule = set(
        contribution_dates(
            price["date"],
            start_date=start,
            end_date=end,
            buy_day=buy_day,
        )
    )
    dividend_events = dividends.copy()
    if dividend_events.empty:
        dividend_events = pd.DataFrame(
            columns=["record_date", "ex_date", "cash_dividend_per_share"]
        )
    dividend_events["record_date"] = pd.to_datetime(
        dividend_events["record_date"], errors="coerce"
    )
    dividend_events["ex_date"] = pd.to_datetime(
        dividend_events["ex_date"], errors="coerce"
    )
    dividend_events = dividend_events[
        dividend_events["ex_date"].between(start, end)
    ].copy()
    grouped_dividends = {
        date: group for date, group in dividend_events.groupby("ex_date")
    }

    cash = 0.0
    shares = 0
    cumulative_contribution = 0.0
    total_dividend = 0.0
    total_purchase_cost = 0.0
    snapshots: dict[pd.Timestamp, int] = {}
    transactions: list[dict[str, Any]] = []
    account_rows: list[dict[str, Any]] = []
    external_cashflows: list[float] = []
    external_dates: list[pd.Timestamp] = []

    def execute_buy(date: pd.Timestamp, close: float, trade_type: str) -> None:
        nonlocal cash, shares, total_purchase_cost
        buy_shares = int(cash // (close * lot_size)) * lot_size
        buy_amount = buy_shares * close
        cash -= buy_amount
        shares += buy_shares
        total_purchase_cost += buy_amount
        transactions.append(
            {
                "symbol": symbol,
                "name": name,
                "horizon": f"{horizon_years}Y",
                "date": date,
                "trade_type": trade_type,
                "buy_price": close,
                "buy_shares": buy_shares,
                "buy_amount": buy_amount,
                "fee": 0.0,
                "remaining_cash": cash,
                "cumulative_shares": shares,
                "cumulative_contribution": cumulative_contribution,
                "repo_interest_accrued": 0.0,
                "dividend_received": 0.0,
                "dividend_tax": 0.0,
            }
        )

    for row in price.itertuples(index=False):
        date = pd.Timestamp(row.date)
        close = float(row.close)
        dividend_received_today = 0.0

        if date in grouped_dividends:
            for event in grouped_dividends[date].itertuples(index=False):
                record_date = (
                    pd.Timestamp(event.record_date)
                    if pd.notna(event.record_date)
                    else date - pd.Timedelta(days=1)
                )
                eligible_shares = _shares_on_or_before(snapshots, record_date)
                dividend_cash = (
                    eligible_shares * float(event.cash_dividend_per_share)
                )
                cash += dividend_cash
                total_dividend += dividend_cash
                dividend_received_today += dividend_cash
            if dividend_reinvest and dividend_received_today > 0:
                execute_buy(date, close, "dividend_reinvest")
                transactions[-1]["dividend_received"] = dividend_received_today

        if date in schedule:
            cash += monthly_amount
            cumulative_contribution += monthly_amount
            external_cashflows.append(-monthly_amount)
            external_dates.append(date)
            execute_buy(date, close, "monthly_contribution")

        snapshots[date] = shares
        market_value = shares * close
        asset = market_value + cash
        account_profit_rate = (
            asset / cumulative_contribution - 1.0
            if cumulative_contribution > 0 else np.nan
        )
        account_rows.append(
            {
                "symbol": symbol,
                "name": name,
                "horizon": f"{horizon_years}Y",
                "date": date,
                "close": close,
                "shares": shares,
                "cash": cash,
                "market_value": market_value,
                "account_asset": asset,
                "cumulative_contribution": cumulative_contribution,
                "account_profit_rate": account_profit_rate,
                "dividend_received": dividend_received_today,
            }
        )

    account = pd.DataFrame(account_rows)
    transaction_frame = pd.DataFrame(transactions)
    ending_market_value = float(account["market_value"].iloc[-1])
    ending_asset = float(account["account_asset"].iloc[-1])
    ending_cash = float(account["cash"].iloc[-1])
    ending_shares = int(account["shares"].iloc[-1])
    external_cashflows.append(ending_asset)
    external_dates.append(end)
    annualized_return = xirr(external_cashflows, external_dates)

    total_history = build_total_return_history(price, dividend_events)
    volatility = float(
        total_history["daily_total_return"].iloc[1:].std(ddof=1) * math.sqrt(252)
    )
    average_buy_price = (
        total_purchase_cost / ending_shares if ending_shares else np.nan
    )
    current_profit_rate = (
        float(price["close"].iloc[-1]) / average_buy_price - 1.0
        if average_buy_price and not np.isnan(average_buy_price) else np.nan
    )
    total_return = (
        ending_asset / cumulative_contribution - 1.0
        if cumulative_contribution else np.nan
    )
    max_loss = float(account["account_profit_rate"].min())
    max_drawdown = float(total_history["drawdown"].min())
    max_loss_duration = _max_loss_duration_days(account)
    strategy_max_dd = _strategy_max_drawdown(account)

    summary = {
        "symbol": symbol,
        "name": name,
        "horizon": f"{horizon_years}Y",
        "requested_start_date": requested_start,
        "start_date": start,
        "end_date": end,
        "listing_date": listing,
        "full_horizon": full_horizon,
        "contribution_months": len(schedule),
        "total_contribution": cumulative_contribution,
        "ending_shares": ending_shares,
        "ending_cash": ending_cash,
        "ending_market_value": ending_market_value,
        "ending_asset": ending_asset,
        "total_dividend": total_dividend,
        "total_return": total_return if full_horizon else np.nan,
        "xirr": annualized_return if full_horizon else np.nan,
        "average_buy_price": average_buy_price if full_horizon else np.nan,
        "current_profit_rate": current_profit_rate if full_horizon else np.nan,
        "max_drawdown": max_drawdown if full_horizon else np.nan,
        "max_loss_vs_contribution": max_loss if full_horizon else np.nan,
        "strategy_max_drawdown": strategy_max_dd if full_horizon else np.nan,
        "max_loss_duration_days": max_loss_duration if full_horizon else np.nan,
        "volatility": volatility if full_horizon else np.nan,
        "dividend_reinvest": dividend_reinvest,
    }
    return BacktestOutput(
        summary=summary,
        transactions=transaction_frame,
        account_history=account,
        total_return_history=total_history.assign(
            symbol=symbol, name=name, horizon=f"{horizon_years}Y"
        ),
    )

### 滚动窗口函数

本组函数实现滚动窗口回测，评估定投策略对起点的敏感性。

- `run_rolling_windows`：对每个结束日调用 `simulate_bank_dca`，返回轻量级汇总（XIRR/total_return/full_horizon），不保留完整账户历史以节省内存。
- `compute_repo_rolling_xirr`：逆回购定投滚动 XIRR，每月投入按 204001 日利率逐日计息，期末一次性取出。
- `compute_rolling_winrates`：滚动窗口胜率汇总，对每个 (symbol, horizon) 计算 XIRR 为正、跑赢逆回购/沪深300/中证500 的比例；少于 12 个可比起点的标记 NaN。

In [ ]:
def run_rolling_windows(
    *,
    symbol: str,
    name: str,
    prices: pd.DataFrame,
    dividends: pd.DataFrame,
    listing_date: Any,
    end_dates: Iterable[Any],
    horizon_years: int,
    monthly_amount: float = 5000.0,
    buy_day: int = 1,
    lot_size: int = 100,
    dividend_reinvest: bool = True,
) -> pd.DataFrame:
    """对每个 end_date 调用 simulate_bank_dca，返回轻量级汇总 DataFrame。

    用于 Task 4 滚动窗口胜率计算。只保留 XIRR、total_return、full_horizon
    与窗口起止日，不保留交易流水与账户历史，避免内存膨胀。
    """
    symbol = normalize_symbol(symbol)
    listing = pd.Timestamp(listing_date)
    price = prices.copy()
    price["date"] = pd.to_datetime(price["date"])

    rows: list[dict[str, Any]] = []
    for end_date in end_dates:
        end_ts = pd.Timestamp(end_date)
        requested_start = end_ts - pd.DateOffset(years=horizon_years)
        if listing > end_ts:
            continue
        window = price[
            price["date"].between(
                min(requested_start, listing), end_ts
            )
        ]
        if window.empty:
            continue
        try:
            output = simulate_bank_dca(
                symbol=symbol,
                name=name,
                prices=price,
                dividends=dividends,
                listing_date=listing,
                as_of_date=end_ts.strftime("%Y-%m-%d"),
                horizon_years=horizon_years,
                monthly_amount=monthly_amount,
                buy_day=buy_day,
                lot_size=lot_size,
                dividend_reinvest=dividend_reinvest,
            )
        except (ValueError, RuntimeError):
            continue
        rows.append(
            {
                "symbol": symbol,
                "name": name,
                "end_date": end_ts,
                "horizon_years": horizon_years,
                "xirr": output.summary["xirr"],
                "total_return": output.summary["total_return"],
                "full_horizon": output.summary["full_horizon"],
                "start_date": output.summary["start_date"],
            }
        )
    return pd.DataFrame(rows)


def compute_repo_rolling_xirr(
    *,
    repo_rates: pd.DataFrame,
    end_dates: Iterable[Any],
    horizon_years: int,
    monthly_amount: float = 5000.0,
    buy_day: int = 1,
) -> pd.DataFrame:
    """对每个 end_date 计算 horizon_years 年逆回购定投 XIRR。

    口径与定投剩余现金计息一致：每月投入固定金额，按 204001 日利率逐日计息，
    期末一次性取出。repo_rates 需包含 date 与 rate_pct 列。
    """
    if repo_rates.empty or "rate_pct" not in repo_rates.columns:
        return pd.DataFrame()
    rates = repo_rates.copy()
    rates["date"] = pd.to_datetime(rates["date"])
    rates = rates.sort_values("date").drop_duplicates("date", keep="last")

    rows: list[dict[str, Any]] = []
    for end_date in end_dates:
        end_ts = pd.Timestamp(end_date)
        start_ts = end_ts - pd.DateOffset(years=horizon_years)
        window = rates[rates["date"].between(start_ts, end_ts)]
        if window.empty:
            continue

        contribution_dates_list: list[pd.Timestamp] = []
        for period in pd.period_range(
            start_ts.to_period("M"), end_ts.to_period("M"), freq="M"
        ):
            target = period.start_time + pd.Timedelta(days=buy_day - 1)
            candidates = window[window["date"] >= target]
            if not candidates.empty:
                contribution_dates_list.append(
                    pd.Timestamp(candidates["date"].iloc[0])
                )

        if len(contribution_dates_list) < 2:
            continue

        cashflows: list[float] = []
        cf_dates: list[pd.Timestamp] = []
        for md in contribution_dates_list:
            cashflows.append(-monthly_amount)
            cf_dates.append(md)

        cash = 0.0
        for _, row in window.iterrows():
            rate = float(row["rate_pct"]) / 100.0 / 365.0
            cash = cash * (1.0 + rate)
            if pd.Timestamp(row["date"]) in set(contribution_dates_list):
                cash += monthly_amount

        cashflows.append(cash)
        cf_dates.append(end_ts)
        xirr_val = xirr(cashflows, cf_dates)
        rows.append(
            {
                "symbol": "GC001",
                "name": "国债逆回购",
                "end_date": end_ts,
                "horizon_years": horizon_years,
                "xirr": xirr_val,
                "total_return": cash / (monthly_amount * len(contribution_dates_list)) - 1.0,
                "full_horizon": len(contribution_dates_list) >= horizon_years * 12,
                "start_date": start_ts,
            }
        )
    return pd.DataFrame(rows)


def compute_rolling_winrates(
    rolling_results: pd.DataFrame,
    benchmark_results: dict[str, pd.DataFrame],
    *,
    min_comparable_windows: int = 12,
) -> pd.DataFrame:
    """汇总滚动窗口胜率。

    对每个 (symbol, horizon_years) 计算：
    - XIRR 为正的比例
    - 跑赢逆回购的比例
    - 跑赢沪深300全收益的比例
    - 跑赢中证500全收益的比例

    少于 min_comparable_windows 个可比起点时胜率标记为 NaN。
    benchmark_results 的 key 为基准名（GC001 / H00300 / H00905）。
    """
    if rolling_results.empty:
        return pd.DataFrame()

    rows: list[dict[str, Any]] = []
    for (symbol, name, horizon), group in rolling_results.groupby(
        ["symbol", "name", "horizon_years"]
    ):
        comparable = group[group["full_horizon"]]
        window_count = len(comparable)
        if window_count < min_comparable_windows:
            rows.append(
                {
                    "symbol": symbol,
                    "name": name,
                    "horizon_years": int(horizon),
                    "window_count": window_count,
                    "comparable_windows": window_count,
                    "xirr_positive_rate": np.nan,
                    "beat_repo_rate": np.nan,
                    "beat_csi300_rate": np.nan,
                    "beat_csi500_rate": np.nan,
                }
            )
            continue

        xirr_positive = (comparable["xirr"] > 0).sum()
        win_rates: dict[str, float] = {
            "xirr_positive_rate": xirr_positive / window_count,
        }

        for bench_name, bench_key in [
            ("GC001", "beat_repo_rate"),
            ("H00300", "beat_csi300_rate"),
            ("H00905", "beat_csi500_rate"),
        ]:
            if bench_name not in benchmark_results or benchmark_results[bench_name].empty:
                win_rates[bench_key] = np.nan
                continue
            bench = benchmark_results[bench_name]
            bench_horizon = bench[bench["horizon_years"] == horizon]
            merged = comparable.merge(
                bench_horizon[["end_date", "xirr"]].rename(
                    columns={"xirr": "benchmark_xirr"}
                ),
                on="end_date",
                how="inner",
            )
            if len(merged) < min_comparable_windows:
                win_rates[bench_key] = np.nan
            else:
                win_rates[bench_key] = (
                    (merged["xirr"] > merged["benchmark_xirr"]).sum() / len(merged)
                )

        rows.append(
            {
                "symbol": symbol,
                "name": name,
                "horizon_years": int(horizon),
                "window_count": len(group),
                "comparable_windows": window_count,
                **win_rates,
            }
        )

    return pd.DataFrame(rows)

### 指数行情工具

本组函数用于补齐基准指数（沪深300全收益 H00300、中证500全收益 H00905）行情，仅在 Task 1 未生成指数行情时调用。

- `direct_domains`：上下文管理器，让东方财富域名直连。
- `call_akshare`：调用 AKShare 接口，局部关闭进度条。
- `to_frame`：将接口返回值统一转为 DataFrame。
- `_normalize_history`：将行情返回值标准化为统一字段。
- `normalize_symbol`：从任意输入提取 6 位证券代码。
- `market_prefix`：返回交易所前缀。

In [ ]:
_BLOCKED_ERROR_NAMES = {
    "ConnectionError",
    "ConnectTimeout",
    "ProxyError",
    "ReadTimeout",
    "RemoteDisconnected",
    "Timeout",
}


_BLOCKED_MESSAGE_PATTERNS = (
    "403",
    "429",
    "connection aborted",
    "connection refused",
    "connection reset",
    "max retries exceeded",
    "remote end closed",
    "timed out",
    "too many requests",
)


_SECRET_PATTERN = re.compile(r"(?i)(token|api[_-]?key|authorization|cookie)=([^&\s]+)")


_CREDENTIAL_URL_PATTERN = re.compile(r"(https?://)([^/@\s]+)@")


def sanitize_error(error: BaseException | str, limit: int = 240) -> str:
    """压缩错误信息，并移除潜在 Token、Cookie 或代理凭据。"""
    text = str(error).replace("\r", " ").replace("\n", " ")
    text = _SECRET_PATTERN.sub(r"\1=***", text)
    text = _CREDENTIAL_URL_PATTERN.sub(r"\1***@", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text[:limit]


def is_blocked_error(error_name: str, error_message: str) -> bool:
    if error_name in _BLOCKED_ERROR_NAMES:
        return True
    lowered = error_message.lower()
    return any(pattern in lowered for pattern in _BLOCKED_MESSAGE_PATTERNS)


@contextmanager
def direct_domains(*domains: str):
    """仅让指定域名在当前调用期间直连，并精确恢复 NO_PROXY。

    不删除 HTTP_PROXY/HTTPS_PROXY，也不设置通配符，因此无关域名继续遵循
    用户原有代理设置；该环境修改仅存在于当前 Python 进程。
    """
    no_proxy_keys = ("NO_PROXY", "no_proxy")
    original = {key: os.environ.get(key) for key in no_proxy_keys}
    entries: list[str] = []
    for value in original.values():
        if value:
            entries.extend(item.strip() for item in value.split(",") if item.strip())
    entries.extend(domain.strip() for domain in domains if domain.strip())
    bypass = ",".join(dict.fromkeys(entries))
    try:
        for key in no_proxy_keys:
            os.environ[key] = bypass
        yield
    finally:
        for key in no_proxy_keys:
            os.environ.pop(key, None)
        for key, value in original.items():
            if value is not None:
                os.environ[key] = value


def _iter_without_progress(iterable: Iterable[Any], *args: Any, **kwargs: Any):
    return iterable


def call_akshare(function: Callable[..., Any], *args: Any, **kwargs: Any) -> Any:
    """调用 AKShare，并局部关闭依赖 ipywidgets 的 Notebook 进度条。"""
    module = importlib.import_module(function.__module__)
    if hasattr(module, "get_tqdm"):
        with patch.object(
            module,
            "get_tqdm",
            return_value=_iter_without_progress,
        ):
            return function(*args, **kwargs)
    return function(*args, **kwargs)


def to_frame(value: Any) -> pd.DataFrame:
    """将常见接口返回值转换为可计数的 DataFrame，不改变原始对象。"""
    if isinstance(value, pd.DataFrame):
        return value.copy()
    if isinstance(value, pd.Series):
        return value.to_frame().T
    if value is None:
        return pd.DataFrame()
    if isinstance(value, list):
        return pd.DataFrame(value)
    if isinstance(value, tuple):
        return pd.DataFrame(list(value))
    if isinstance(value, dict):
        try:
            return pd.DataFrame(value)
        except (ValueError, TypeError):
            return pd.DataFrame([value])
    return pd.DataFrame({"value": [value]})


def normalize_symbol(value: Any) -> str:
    digits = "".join(c for c in str(value) if c.isdigit())
    return digits[-6:].zfill(6)


def market_prefix(symbol: str) -> str:
    s = normalize_symbol(symbol)
    if s.startswith(("6", "9")):
        return "sh"
    if s.startswith(("4", "8")):
        return "bj"
    return "sz"


PRICE_COLUMNS = (
    "date", "symbol", "open", "high", "low", "close", "volume", "amount",
    "adjustment", "source",
)


def _normalize_history(
    value: Any,
    *,
    symbol: str,
    source: str,
    adjustment: str,
) -> pd.DataFrame:
    frame = to_frame(value).rename(
        columns={
            "日期": "date",
            "datetime": "date",
            "开盘": "open",
            "最高": "high",
            "最低": "low",
            "收盘": "close",
            "成交量": "volume",
            "成交额": "amount",
        }
    ).copy()
    required = ("date", "open", "high", "low", "close")
    missing = [column for column in required if column not in frame]
    if missing:
        raise ValueError(f"{source} 缺少行情字段: {missing}")
    for column in ("volume", "amount"):
        if column not in frame:
            frame[column] = np.nan
    frame["date"] = pd.to_datetime(frame["date"], errors="coerce")
    for column in ("open", "high", "low", "close", "volume", "amount"):
        frame[column] = pd.to_numeric(frame[column], errors="coerce")
    frame["symbol"] = normalize_symbol(symbol)
    frame["adjustment"] = adjustment
    frame["source"] = source
    frame = frame.dropna(subset=["date"]).sort_values("date")
    frame = frame.drop_duplicates("date", keep="last").reset_index(drop=True)
    return frame[list(PRICE_COLUMNS)]

## 3. 读取 Task 1 标准表

In [ ]:
security_info = pd.read_csv(DATA_DIR / "security_info.csv", parse_dates=["list_date", "manual_review_as_of"])
bank_universe = security_info[security_info["manual_verified"] | security_info["f10_is_bank"]].copy()
print(f"银行标的池：{len(bank_universe)} 只")

# 读取交易日历以确定滚动窗口结束月
trading_calendar_path = DATA_DIR / "trading_calendar.csv"
if trading_calendar_path.is_file():
    trading_calendar = pd.read_csv(trading_calendar_path, parse_dates=["date"])
    trading_dates = trading_calendar["date"].sort_values().unique()
else:
    dates_set = set()
    for row in bank_universe.itertuples(index=False):
        price_path = PRICE_DIR / f"{row.symbol}_daily_raw.parquet"
        if price_path.is_file():
            prices = pd.read_parquet(price_path)
            dates_set.update(pd.to_datetime(prices["date"]).tolist())
    trading_dates = pd.DatetimeIndex(sorted(dates_set))
    print(f"从行情提取交易日：{len(trading_dates)} 个")

# 构造滚动窗口结束月：每月最后一个交易日
as_of_ts = pd.Timestamp(AS_OF_DATE)
end_dates_all = pd.DatetimeIndex(trading_dates[trading_dates <= as_of_ts])
monthly_end = pd.Series(end_dates_all).groupby(end_dates_all.to_period("M")).max()
rolling_end_dates = pd.DatetimeIndex(monthly_end.values)
print(f"滚动窗口结束月数：{len(rolling_end_dates)}")

## 4. 银行股滚动窗口回测

In [ ]:
bank_rolling_rows = []
for row in bank_universe.itertuples(index=False):
    symbol = row.symbol
    name = row.name
    price_path = PRICE_DIR / f"{symbol}_daily_raw.parquet"
    dividend_path = DIVIDEND_DIR / f"{symbol}_dividend.parquet"
    if not price_path.is_file():
        continue
    prices = pd.read_parquet(price_path)
    prices["date"] = pd.to_datetime(prices["date"])
    dividends = pd.read_parquet(dividend_path) if dividend_path.is_file() else pd.DataFrame(
        columns=["ex_date", "record_date", "cash_dividend_per_share"]
    )
    if not dividends.empty:
        dividends["ex_date"] = pd.to_datetime(dividends["ex_date"])
        dividends["record_date"] = pd.to_datetime(dividends["record_date"])

    for horizon in HORIZONS:
        result = run_rolling_windows(
            symbol=symbol, name=name, prices=prices, dividends=dividends,
            listing_date=row.list_date, end_dates=rolling_end_dates,
            horizon_years=horizon, monthly_amount=MONTHLY_AMOUNT,
            buy_day=BUY_DAY, lot_size=LOT_SIZE, dividend_reinvest=DIVIDEND_REINVEST,
        )
        bank_rolling_rows.append(result)
        if not result.empty:
            print(f"{symbol} {name} {horizon}Y: {len(result)} 窗口, 可比 {result['full_horizon'].sum()}")

bank_rolling = pd.concat([r for r in bank_rolling_rows if not r.empty], ignore_index=True) if bank_rolling_rows else pd.DataFrame()
print(f"\n银行滚动窗口合计：{len(bank_rolling)} 行")

## 5. 基准滚动窗口回测

基准：国债逆回购 GC001、沪深300全收益 H00300、中证500全收益 H00905。指数行情若 Task 1 未生成，本任务补齐。

In [ ]:
# 5.1 国债逆回购 GC001
repo_path = DATA_DIR / "repo_rates.csv"
if repo_path.is_file():
    repo_rates = pd.read_csv(repo_path, parse_dates=["date"])
else:
    repo_rates = pd.DataFrame(columns=["date", "rate_pct"])
    print("逆回购表为空 schema，跳过 GC001 基准")

benchmark_results = {}
if not repo_rates.empty:
    repo_rolling_rows = []
    for horizon in HORIZONS:
        result = compute_repo_rolling_xirr(
            repo_rates=repo_rates, end_dates=rolling_end_dates,
            horizon_years=horizon, monthly_amount=MONTHLY_AMOUNT, buy_day=BUY_DAY,
        )
        repo_rolling_rows.append(result)
    repo_rolling = pd.concat([r for r in repo_rolling_rows if not r.empty], ignore_index=True) if repo_rolling_rows else pd.DataFrame()
    benchmark_results["GC001"] = repo_rolling
    print(f"GC001 滚动窗口：{len(repo_rolling)} 行")
else:
    print("GC001 基准不可用（逆回购表为空）")

In [ ]:
# 5.2 指数全收益 H00300 / H00905
for index_code, index_name in [("H00300", "沪深300全收益"), ("H00905", "中证500全收益")]:
    index_path = INDEX_DIR / f"{index_code}_daily.parquet"
    if not index_path.is_file():
        try:
            import akshare as ak
            with direct_domains("eastmoney.com"):
                raw = call_akshare(ak.stock_zh_index_daily_em, symbol=index_code)
            index_prices = _normalize_history(raw, symbol=index_code, source="AKShare-东财", adjustment="raw")
            index_prices.to_parquet(index_path, index=False)
            print(f"补齐 {index_code} 行情：{len(index_prices)} 行")
        except Exception as error:
            print(f"{index_code} 行情补齐失败：{type(error).__name__}: {error}")
            continue
    else:
        index_prices = pd.read_parquet(index_path)
        index_prices["date"] = pd.to_datetime(index_prices["date"])

    index_rolling_rows = []
    for horizon in HORIZONS:
        result = run_rolling_windows(
            symbol=index_code, name=index_name,
            prices=index_prices, dividends=pd.DataFrame(),
            listing_date=index_prices["date"].min(),
            end_dates=rolling_end_dates, horizon_years=horizon,
            monthly_amount=MONTHLY_AMOUNT, buy_day=BUY_DAY,
            lot_size=LOT_SIZE, dividend_reinvest=False,
        )
        index_rolling_rows.append(result)
    index_rolling = pd.concat([r for r in index_rolling_rows if not r.empty], ignore_index=True) if index_rolling_rows else pd.DataFrame()
    benchmark_results[index_code] = index_rolling
    print(f"{index_code} 滚动窗口：{len(index_rolling)} 行")

## 6. 滚动窗口胜率汇总

对每只银行 × 每档期限计算 4 项胜率。少于 12 个可比起点的标记 `N/A`。

In [ ]:
winrate_df = compute_rolling_winrates(
    bank_rolling, benchmark_results,
    min_comparable_windows=MIN_COMPARABLE_WINDOWS,
)
display(winrate_df.head(20))
print(f"胜率汇总：{len(winrate_df)} 行")

## 7. 图表

4 张图：三档期限胜率柱状图、胜率热力图、滚动 XIRR 分布箱线图、基准 vs 银行股胜率对比。

In [ ]:
# 图1：三档期限胜率柱状图（固定样本 × 4 胜率指标）
fixed_symbols = [s for s, _ in FIXED_SAMPLE]
fixed_winrate = winrate_df[winrate_df["symbol"].isin(fixed_symbols)].copy()

if not fixed_winrate.empty:
    winrate_cols = ["xirr_positive_rate", "beat_repo_rate", "beat_csi300_rate", "beat_csi500_rate"]
    winrate_labels = ["XIRR 为正", "跑赢逆回购", "跑赢沪深300", "跑赢中证500"]
    fig, axes = plt.subplots(len(HORIZONS), 1, figsize=(12, 4 * len(HORIZONS)), sharey=True)
    for i, horizon in enumerate(HORIZONS):
        ax = axes[i] if len(HORIZONS) > 1 else axes
        subset = fixed_winrate[fixed_winrate["horizon_years"] == horizon]
        if subset.empty:
            continue
        x = range(len(subset))
        width = 0.2
        for j, (col, label) in enumerate(zip(winrate_cols, winrate_labels)):
            values = subset[col].fillna(0).values
            ax.bar([xi + j * width for xi in x], values, width, label=label)
        ax.set_xticks([xi + 1.5 * width for xi in x])
        ax.set_xticklabels(subset["name"].values, rotation=30)
        ax.set_title(f"{horizon}Y 滚动窗口胜率")
        ax.set_ylabel("胜率")
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3, axis="y")
    plt.suptitle("固定样本银行滚动窗口胜率")
    plt.tight_layout()
    plt.savefig(CHART_DIR / "01_winrate_bars.png", dpi=120)
    plt.show()
else:
    print("固定样本胜率数据为空，跳过图1")

In [ ]:
# 图2：胜率热力图（5 只样本 × 3 期限 × 4 胜率指标）
if not fixed_winrate.empty:
    winrate_cols = ["xirr_positive_rate", "beat_repo_rate", "beat_csi300_rate", "beat_csi500_rate"]
    winrate_labels = ["XIRR>0", ">逆回购", ">沪深300", ">中证500"]

    fig, axes = plt.subplots(1, len(HORIZONS), figsize=(5 * len(HORIZONS), 5))
    for i, horizon in enumerate(HORIZONS):
        ax = axes[i] if len(HORIZONS) > 1 else axes
        subset = fixed_winrate[fixed_winrate["horizon_years"] == horizon]
        if subset.empty:
            continue
        matrix = subset.set_index("name")[winrate_cols].reindex([n for _, n in FIXED_SAMPLE])
        im = ax.imshow(matrix.T.values, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
        ax.set_xticks(range(len(matrix)))
        ax.set_xticklabels(matrix.index, rotation=30)
        ax.set_yticks(range(len(winrate_labels)))
        ax.set_yticklabels(winrate_labels)
        ax.set_title(f"{horizon}Y")
        for r in range(matrix.shape[0]):
            for c in range(matrix.shape[1]):
                val = matrix.iloc[r, c]
                if pd.notna(val):
                    ax.text(c, r, f"{val:.0%}", ha="center", va="center", fontsize=9)
        plt.colorbar(im, ax=ax, fraction=0.046)
    plt.suptitle("固定样本银行胜率热力图")
    plt.tight_layout()
    plt.savefig(CHART_DIR / "02_winrate_heatmap.png", dpi=120)
    plt.show()
else:
    print("固定样本胜率数据为空，跳过图2")

In [ ]:
# 图3：滚动 XIRR 分布箱线图（5 只样本在不同窗口结束月的 XIRR）
fixed_rolling = bank_rolling[bank_rolling["symbol"].isin(fixed_symbols) & bank_rolling["full_horizon"]].copy()

if not fixed_rolling.empty:
    fig, axes = plt.subplots(1, len(HORIZONS), figsize=(6 * len(HORIZONS), 5), sharey=True)
    for i, horizon in enumerate(HORIZONS):
        ax = axes[i] if len(HORIZONS) > 1 else axes
        subset = fixed_rolling[fixed_rolling["horizon_years"] == horizon]
        box_data = [subset[subset["symbol"] == s]["xirr"].dropna().values for s, _ in FIXED_SAMPLE]
        box_data = [d for d in box_data if len(d) > 0]
        if box_data:
            ax.boxplot(box_data, labels=[n for s, n in FIXED_SAMPLE if s in subset["symbol"].values][:len(box_data)])
            ax.tick_params(axis="x", rotation=30)
        ax.axhline(0, color="red", linestyle="--", alpha=0.5)
        ax.set_title(f"{horizon}Y")
        ax.set_ylabel("XIRR")
        ax.grid(True, alpha=0.3, axis="y")
    plt.suptitle("固定样本银行滚动 XIRR 分布")
    plt.tight_layout()
    plt.savefig(CHART_DIR / "03_xirr_distribution.png", dpi=120)
    plt.show()
else:
    print("固定样本滚动窗口数据为空，跳过图3")

In [ ]:
# 图4：基准 vs 银行股胜率对比
bench_winrate_rows = []
for bench_name, bench_df in benchmark_results.items():
    if bench_df.empty:
        continue
    for horizon in HORIZONS:
        bench_horizon = bench_df[bench_df["horizon_years"] == horizon]
        comparable = bench_horizon[bench_horizon["full_horizon"]]
        if len(comparable) < MIN_COMPARABLE_WINDOWS:
            continue
        bench_winrate_rows.append({
            "name": bench_name,
            "horizon_years": horizon,
            "xirr_positive_rate": (comparable["xirr"] > 0).sum() / len(comparable),
            "comparable_windows": len(comparable),
        })

bench_winrate = pd.DataFrame(bench_winrate_rows)
if not bench_winrate.empty:
    fig, axes = plt.subplots(1, len(HORIZONS), figsize=(6 * len(HORIZONS), 5), sharey=True)
    for i, horizon in enumerate(HORIZONS):
        ax = axes[i] if len(HORIZONS) > 1 else axes
        bank_subset = fixed_winrate[fixed_winrate["horizon_years"] == horizon]
        bank_mean = bank_subset["xirr_positive_rate"].mean() if not bank_subset.empty else 0
        bench_subset = bench_winrate[bench_winrate["horizon_years"] == horizon]
        if not bench_subset.empty:
            names = ["银行股均值"] + bench_subset["name"].tolist()
            values = [bank_mean] + bench_subset["xirr_positive_rate"].tolist()
            ax.bar(names, values, color=["steelblue"] + ["orange"] * len(bench_subset))
            ax.set_title(f"{horizon}Y XIRR 为正胜率")
            ax.set_ylabel("胜率")
            ax.tick_params(axis="x", rotation=30)
            ax.grid(True, alpha=0.3, axis="y")
    plt.suptitle("基准 vs 银行股胜率对比")
    plt.tight_layout()
    plt.savefig(CHART_DIR / "04_benchmark_comparison.png", dpi=120)
    plt.show()
else:
    print("基准胜率数据为空，跳过图4")

## 8. 保存结果

In [ ]:
rolling_path = RESULT_DIR / "task4_rolling_windows.csv"
winrate_path = RESULT_DIR / "task4_winrates.csv"

bank_rolling.to_csv(rolling_path, index=False, encoding="utf-8-sig")
winrate_df.to_csv(winrate_path, index=False, encoding="utf-8-sig")

assert len(bank_rolling) > 0, "滚动窗口结果为空"
assert len(winrate_df) > 0, "胜率汇总为空"
chart_files = list(CHART_DIR.glob("*.png"))
assert len(chart_files) >= 3, f"应生成至少 3 张图，实际 {len(chart_files)} 张"
print(f"Task 4 验收通过")
print(f"- {len(bank_rolling)} 行滚动窗口结果")
print(f"- {len(winrate_df)} 行胜率汇总")
print(f"- {len(chart_files)} 张图表生成")
print(f"- 基准：{', '.join(benchmark_results.keys())}")